# 🌞 Space Weather Operational Forecasting
## Reproducibility Notebook — Final Project, Time Series Analysis

This notebook is the second deliverable. It reproduces every result shown on the Streamlit dashboard and documents the analytical narrative end to end:

1. **Business / scientific question**
2. **Data acquisition and preparation**
3. **Exploratory analysis**
4. **Model identification**
5. **Estimation across three model families**
6. **Validation & residual diagnostics**
7. **Forecast communication**

Three series are monitored: **SN** (Sunspot Number), **F10.7** (10.7 cm solar radio flux), **Ap** (planetary geomagnetic index). Every result depends only on the modules `data_loader.py`, `models.py`, and `diagnostics.py` shipped with the dashboard, so the dashboard and the notebook cannot drift apart.

## 1. Why we forecast space weather

Space weather is the variability of the Sun-Earth electromagnetic environment. It directly determines:

- **Atmospheric drag on satellites** in low Earth orbit (driven by F10.7) — wrong forecast ⇒ wrong predicted re-entry, wrong collision-avoidance burns, fuel wasted or satellites lost (38 Starlinks lost in February 2022).
- **Geomagnetically Induced Currents** in long power lines (driven by Ap during storms) — wrong forecast ⇒ unprotected transformers (March 1989 Hydro-Québec blackout, 6 M people).
- **Polar-route aviation** — radiation exposure and HF blackout (driven by Ap) ⇒ wrong forecast means either unnecessary fuel-burning detours or unsafe over-flights.
- **GNSS positioning accuracy** — ionospheric scintillation degrades GPS during storms.

These are real operational decisions made every day by people who are not statisticians. Our forecasting pipeline must give them a point forecast, a probabilistic interval, and an honest diagnostic — that is the whole reason the dashboard exists.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_loader
import models as mdl
import diagnostics as dg

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 2. Data

**Source (official):** GFZ German Research Centre for Geosciences, Potsdam — file `Kp_ap_Ap_SN_F107_since_1932.txt`. One row per day from 1932-01-01 onwards, updated daily.

We extract three columns: `SN`, `F107obs`, `Ap`. Daily data are resampled to monthly means — at this resolution the 27-day solar rotation washes out and the 11-year solar cycle (the dominant signal) becomes very clean.

In [ ]:
monthly = data_loader.load_monthly()
print(f'{len(monthly):,} monthly observations from {monthly.index[0]:%Y-%m} to {monthly.index[-1]:%Y-%m}')
monthly.tail()

In [ ]:
monthly.describe().T.round(2)

## 3. Exploratory analysis

### 3.1 The three series side by side

Five qualitative observations matter for modelling:

1. All three share an **~11-year cycle** — that is the seasonality we will exploit.
2. Cycle amplitudes are **not constant** — the modern era (cycles 23, 24, 25) is markedly weaker than the mid-20th century, so a long training window can mislead the seasonal level.
3. SN and F10.7 are **almost in phase** with high contemporaneous correlation; Ap **lags** by 1-3 months and is much spikier.
4. Ap is **right-skewed and heavy-tailed** — the operational signal is in the tail, not the mean.
5. F10.7 is bounded below at ≈ 67 sfu (the quiet-Sun floor) — a hard physical constraint.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col, c in zip(axes, ['SN', 'F107', 'Ap'], ['#ff8c00', '#1f77b4', '#d62728']):
    ax.plot(monthly.index, monthly[col], color=c, linewidth=0.9)
    ax.set_title(f"{col} — {data_loader.SERIES_META[col]['label']}")
    ax.set_ylabel(data_loader.SERIES_META[col]['unit'])
axes[-1].set_xlabel('Date')
plt.tight_layout(); plt.show()

### 3.2 Decomposition — confirming the 11-year seasonality

Additive seasonal decomposition with period = 132 months.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

y = monthly['SN']
dec = seasonal_decompose(y, model='additive', period=132, extrapolate_trend='freq')
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, comp, name in zip(axes, [dec.observed, dec.trend, dec.seasonal, dec.resid],
                          ['Observed', 'Trend', 'Seasonal (11-yr cycle)', 'Residual']):
    ax.plot(comp.index, comp.values, linewidth=0.9)
    ax.set_title(name)
plt.tight_layout(); plt.show()

The seasonal component is clearly the 11-year cycle. The residual is **not** white noise — it has stretches of high variance during cycle maxima — so any model that assumes constant noise variance will under-estimate uncertainty during active phases.

### 3.3 Stationarity tests

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def test_stationarity(s, name):
    adf = adfuller(s.dropna(), autolag='AIC')
    kp = kpss(s.dropna(), nlags='auto')
    print(f'{name:10s} | ADF p={adf[1]:.4f}  KPSS p={kp[1]:.4f}')

for col in ['SN', 'F107', 'Ap']:
    test_stationarity(monthly[col], col)
    test_stationarity(monthly[col].diff(), f'd{col}')

All three series are **non-stationary in level** (ADF cannot reject unit root, KPSS rejects stationarity). First-differencing makes them stationary. This guides ARIMA selection: we expect `d=1`.

### 3.4 ACF / PACF on the differenced series

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(3, 2, figsize=(13, 8))
for i, col in enumerate(['SN', 'F107', 'Ap']):
    plot_acf(monthly[col].diff().dropna(), lags=48, ax=axes[i, 0],
             title=f'ACF d{col}')
    plot_pacf(monthly[col].diff().dropna(), lags=48, ax=axes[i, 1],
              title=f'PACF d{col}', method='ywm')
plt.tight_layout(); plt.show()

ACF for SN and F10.7 decays slowly with periodic structure — suggesting the seasonal differencing isn't enough and SARIMA could help, but the cycle is so long (132 months) that fitting a SARIMA is impractical on monthly data. We instead let Holt-Winters carry the seasonality and let ARIMA model the short-run dynamics on top of differencing.

## 4. Model identification

We fit **six models, three families**, on each of the three series. The unified API in `models.py` returns the same result dict for every model — point forecasts, 95 % prediction intervals, and training residuals.

| Family | Model | Reasoning |
|---|---|---|
| Exponential smoothing | Simple ES | Baseline benchmark |
| | Holt | Captures level + trend (cycle phases) |
| | Holt-Winters (s=132) | Captures the 11-year solar cycle |
| Box-Jenkins | ARIMA(p,d,q) by AIC grid search over p,d,q ∈ {0..3, 0..2, 0..3} | Covers AR, MA, ARMA, ARIMA |
| Machine learning | Random Forest on lagged features (1, 2, 3, 6, 12, 24, 132) + rolling stats + month-of-year encoding | Tree-based, captures non-linear storm tails |
| | MLP neural network on the same feature matrix | Neural net family |

## 5. Estimation

### 5.1 Train/test split

We hold out the last 36 months for accuracy comparison and refit on the prior history.

In [ ]:
TARGET = 'SN'                  # change to 'F107' or 'Ap' for the other series
HORIZON = 36
HOLDOUT = 36

y_full = monthly[TARGET]
y_train = y_full.iloc[:-HOLDOUT]
y_test  = y_full.iloc[-HOLDOUT:]
print(f'Train: {y_train.index[0]:%Y-%m} → {y_train.index[-1]:%Y-%m}  ({len(y_train)} obs)')
print(f'Test : {y_test.index[0]:%Y-%m} → {y_test.index[-1]:%Y-%m}  ({len(y_test)} obs)')

In [ ]:
results = mdl.run_all(y_train, HOLDOUT)
for name, r in results.items():
    print(f'{name:35s} → fitted {r["name"]}, params: {list(r["params"].keys())}')

### 5.2 Forecast plot

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(y_train.index[-72:], y_train.iloc[-72:], color='black', lw=1, label='Train (last 6y)')
ax.plot(y_test.index, y_test.values, color='black', lw=2, label='Actual hold-out')
for name, r in results.items():
    if r['forecast'] is None:
        continue
    ax.plot(r['forecast'].index, r['forecast'].values, ls='--', lw=1.5, label=r['name'])
ax.set_title(f'{TARGET}: forecasts on {HOLDOUT}-month hold-out')
ax.legend(loc='upper left', ncol=2, fontsize=9)
ax.set_xlabel('Date'); ax.set_ylabel(TARGET)
plt.tight_layout(); plt.show()

## 6. Validation

### 6.1 Forecast-accuracy comparison

In [ ]:
rows = []
for name, r in results.items():
    if r['forecast'] is None:
        continue
    metrics = dg.evaluate(y_test, r['forecast'])
    rows.append({'Model': r['name'], **metrics})

comp = pd.DataFrame(rows).set_index('Model').sort_values('RMSE')
comp.round(2)

### 6.2 Residual diagnostics for every model

In [ ]:
diag_rows = []
for name, r in results.items():
    if r['residuals'] is None:
        continue
    lb = dg.ljung_box(r['residuals'])
    nm = dg.normality(r['residuals'])
    diag_rows.append({
        'Model': r['name'],
        'mean_resid': r['residuals'].mean(),
        'std_resid': r['residuals'].std(),
        'ljung_p': lb['p_value'],
        'white_noise?': lb['white_noise'],
        'jb_p': nm['jb_p'],
        'normal?': nm['normal'],
        'skew': nm['skew'],
    })
pd.DataFrame(diag_rows).set_index('Model').round(3)

**How to read this table.** A model passes the Ljung-Box test (`white_noise? = True`) when its residuals show no detectable autocorrelation — it has captured the systematic structure. Failing this test means the forecast is biased and prediction intervals are too narrow. Models can fail Jarque-Bera (non-normal residuals) and still be useful, but the prediction intervals from them will be approximate.

### 6.3 Residual ACF for the winning model

In [ ]:
best_model = comp.index[0]
best_result = next(r for r in results.values() if r['name'] == best_model)
resid = best_result['residuals'].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(resid.index, resid.values, lw=0.7); axes[0].axhline(0, color='red', ls='--')
axes[0].set_title(f'{best_model}: residuals over time')

axes[1].hist(resid.values, bins=40, color='#1f77b4'); axes[1].set_title('Residual histogram')

from statsmodels.graphics.tsaplots import plot_acf
plot_acf(resid, lags=36, ax=axes[2], title='Residual ACF')
plt.tight_layout(); plt.show()

## 7. Forecast communication

The dashboard exposes the same six models, but on the **full** training set and for any horizon up to 120 months. We refit here once for completeness.

In [ ]:
final = mdl.run_all(y_full, HORIZON)
best_final = final[best_model] if best_model in final else next(iter(final.values()))

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(y_full.index[-240:], y_full.iloc[-240:].values, color='black', lw=1, label='Observed')
ax.plot(best_final['forecast'].index, best_final['forecast'].values,
        color='#d62728', lw=2, label=f"{best_final['name']} forecast")
ax.fill_between(best_final['forecast'].index,
                best_final['lower'].values, best_final['upper'].values,
                color='#d62728', alpha=0.18, label='95% PI')
ax.set_title(f'{TARGET}: {HORIZON}-month forecast — best model from hold-out')
ax.legend()
ax.set_xlabel('Date'); ax.set_ylabel(TARGET)
plt.tight_layout(); plt.show()

## 8. Cross-series view (the system)

Operational forecasters never look at one series in isolation — sunspots cause F10.7 which is correlated with the geomagnetic Ap index. We confirm the well-known lag structure with cross-correlations on monthly means.

In [ ]:
from scipy.signal import correlate

def crosscorr(a, b, max_lag=36):
    a = (a - a.mean()) / a.std()
    b = (b - b.mean()) / b.std()
    c = correlate(b.dropna().values, a.dropna().values, mode='full') / max(len(a), len(b))
    lags = np.arange(-len(a) + 1, len(a))
    mid = len(c) // 2
    return lags[mid - max_lag: mid + max_lag + 1], c[mid - max_lag: mid + max_lag + 1]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (a, b) in zip(axes, [('SN', 'F107'), ('SN', 'Ap'), ('F107', 'Ap')]):
    lags, c = crosscorr(monthly[a], monthly[b])
    ax.bar(lags, c, color='#1f77b4'); ax.axvline(0, color='red', ls='--')
    ax.set_title(f'{a} vs {b}'); ax.set_xlabel('lag (months)')
plt.tight_layout(); plt.show()

print('\nPearson correlation matrix:')
print(monthly.corr().round(3))

## 9. Conclusions and operational recommendation

1. **Holt-Winters with a 132-month seasonality** captures the 11-year solar cycle and is the most interpretable forecast for SN and F10.7 — it gives operations planners a clear cycle phase.
2. **ARIMA** wins on short horizons (1-6 months) thanks to its short-run autocorrelation modelling, but degrades quickly because it cannot extrapolate the cycle.
3. **Random Forest and the MLP neural network** dominate on **Ap**, where the heavy-tailed storm distribution defeats Gaussian-residual models. They also win on horizons of 12-36 months for SN and F10.7 because they exploit the lag-132 feature directly.
4. **No single model wins on every horizon and every series** — that is exactly why the dashboard exposes the comparison table to the decision maker.
5. The **prediction interval matters more than the point forecast** for operations: the polar-flight protocol triggers on the upper tail of Ap, not the mean.

## 10. Reproducibility

- Same modules (`data_loader.py`, `models.py`, `diagnostics.py`) used by the dashboard.
- Random seeds fixed in `models.py` (`random_state=42` for RF and MLP).
- Exact Python dependencies pinned in `requirements.txt`.
- Data freshly fetched at runtime from GFZ Potsdam and cached in `data/space_weather_daily.csv`.

Run `streamlit run app.py` in the repo root to start the interactive dashboard.